# 04 — Tree Models

**Objective:** Plan reproducible tree-based model experiments.  
**Owner:** TBD  
**Sprint:** TBD  

> Leakage warning: reserve the test set for final evaluation only.

In [2]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

X, y = fetch_openml(
    data_id=45566,
    as_frame=True,
    return_X_y=True
)

y = y.astype(str).map({
    "False": 0,
    "True": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

X shape: (200000, 200)
y shape: (200000,)

Target distribution:
target
0    0.89951
1    0.10049
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))
print("y_val distribution:")
print(y_val.value_counts(normalize=True))

X_train: (160000, 200)
X_val: (40000, 200)
y_train distribution:
target
0    0.899513
1    0.100487
Name: proportion, dtype: float64
y_val distribution:
target
0    0.8995
1    0.1005
Name: proportion, dtype: float64


In [ ]:
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import load_config

config = load_config()
print(config["project"]["name"])
print(config["project"]["random_state"])

In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score

tree_model = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

tree_model.fit(X_train, y_train)

y_pred = tree_model.predict(X_val)
y_prob = tree_model.predict_proba(X_val)[:, 1]

print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

              precision    recall  f1-score   support

           0       0.93      0.67      0.78     35980
           1       0.16      0.54      0.24      4020

    accuracy                           0.66     40000
   macro avg       0.54      0.61      0.51     40000
weighted avg       0.85      0.66      0.72     40000

ROC-AUC: 0.6365885379937445


## Planned work

Later: compare approved tree estimators under the shared split, cross-validation, metrics, and experiment logging conventions.

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)
rf_prob = rf_model.predict_proba(X_val)[:, 1]

print(classification_report(y_val, rf_pred))
print("ROC-AUC:", roc_auc_score(y_val, rf_prob))

              precision    recall  f1-score   support

           0       0.95      0.84      0.89     35980
           1       0.29      0.58      0.39      4020

    accuracy                           0.82     40000
   macro avg       0.62      0.71      0.64     40000
weighted avg       0.88      0.82      0.84     40000

ROC-AUC: 0.7977387174743292
